In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

# IMPORT LIBRARY

In [4]:
!pip install underthesea

In [5]:
!pip install transformers

In [6]:
from underthesea import word_tokenize,text_normalize
import torch
from transformers import AutoTokenizer,get_linear_schedule_with_warmup
import pandas as pd
import json
import numpy as np
from PIL import Image
from torchvision import transforms
import os
import re

C:\Users\HIEU\Model_Setiment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
sss = 0
torch.manual_seed(sss)
np.random.seed(sss)

# TEXT PRE-PROCESSING

In [9]:
def convert_unicode(text):
  char1252 = 'à|á|ả|ã|ạ|ầ|ấ|ẩ|ẫ|ậ|ằ|ắ|ẳ|ẵ|ặ|è|é|ẻ|ẽ|ẹ|ề|ế|ể|ễ|ệ|ì|í|ỉ|ĩ|ị|ò|ó|ỏ|õ|ọ|ồ|ố|ổ|ỗ|ộ|ờ|ớ|ở|ỡ|ợ|ù|ú|ủ|ũ|ụ|ừ|ứ|ử|ữ|ự|ỳ|ý|ỷ|ỹ|ỵ|À|Á|Ả|Ã|Ạ|Ầ|Ấ|Ẩ|Ẫ|Ậ|Ằ|Ắ|Ẳ|Ẵ|Ặ|È|É|Ẻ|Ẽ|Ẹ|Ề|Ế|Ể|Ễ|Ệ|Ì|Í|Ỉ|Ĩ|Ị|Ò|Ó|Ỏ|Õ|Ọ|Ồ|Ố|Ổ|Ỗ|Ộ|Ờ|Ớ|Ở|Ỡ|Ợ|Ù|Ú|Ủ|Ũ|Ụ|Ừ|Ứ|Ử|Ữ|Ự|Ỳ|Ý|Ỷ|Ỹ|Ỵ'
  charutf8 = 'à|á|ả|ã|ạ|ầ|ấ|ẩ|ẫ|ậ|ằ|ắ|ẳ|ẵ|ặ|è|é|ẻ|ẽ|ẹ|ề|ế|ể|ễ|ệ|ì|í|ỉ|ĩ|ị|ò|ó|ỏ|õ|ọ|ồ|ố|ổ|ỗ|ộ|ờ|ớ|ở|ỡ|ợ|ù|ú|ủ|ũ|ụ|ừ|ứ|ử|ữ|ự|ỳ|ý|ỷ|ỹ|ỵ|À|Á|Ả|Ã|Ạ|Ầ|Ấ|Ẩ|Ẫ|Ậ|Ằ|Ắ|Ẳ|Ẵ|Ặ|È|É|Ẻ|Ẽ|Ẹ|Ề|Ế|Ể|Ễ|Ệ|Ì|Í|Ỉ|Ĩ|Ị|Ò|Ó|Ỏ|Õ|Ọ|Ồ|Ố|Ổ|Ỗ|Ộ|Ờ|Ớ|Ở|Ỡ|Ợ|Ù|Ú|Ủ|Ũ|Ụ|Ừ|Ứ|Ử|Ữ|Ự|Ỳ|Ý|Ỷ|Ỹ|Ỵ'
  char1252 = char1252.split('|')
  charutf8 = charutf8.split('|')

  dic = {}
  for i in range(len(char1252)): dic[char1252[i]] = charutf8[i]
  return re.sub(
      r'à|á|ả|ã|ạ|ầ|ấ|ẩ|ẫ|ậ|ằ|ắ|ẳ|ẵ|ặ|è|é|ẻ|ẽ|ẹ|ề|ế|ể|ễ|ệ|ì|í|ỉ|ĩ|ị|ò|ó|ỏ|õ|ọ|ồ|ố|ổ|ỗ|ộ|ờ|ớ|ở|ỡ|ợ|ù|ú|ủ|ũ|ụ|ừ|ứ|ử|ữ|ự|ỳ|ý|ỷ|ỹ|ỵ|À|Á|Ả|Ã|Ạ|Ầ|Ấ|Ẩ|Ẫ|Ậ|Ằ|Ắ|Ẳ|Ẵ|Ặ|È|É|Ẻ|Ẽ|Ẹ|Ề|Ế|Ể|Ễ|Ệ|Ì|Í|Ỉ|Ĩ|Ị|Ò|Ó|Ỏ|Õ|Ọ|Ồ|Ố|Ổ|Ỗ|Ộ|Ờ|Ớ|Ở|Ỡ|Ợ|Ù|Ú|Ủ|Ũ|Ụ|Ừ|Ứ|Ử|Ữ|Ự|Ỳ|Ý|Ỷ|Ỹ|Ỵ',
      lambda x: dic[x.group()], text
)


In [10]:
class TextNormalize:
    def __init__(self):
        self.vowels_to_ids = {}
        self.vowels_table = [
            ['a', 'à', 'á', 'ả', 'ã', 'ạ', 'a' ],
            ['ă', 'ằ', 'ắ', 'ẳ', 'ẵ', 'ặ', 'aw'],
            ['â', 'ầ', 'ấ', 'ẩ', 'ẫ', 'ậ', 'aa'],
            ['e', 'è', 'é', 'ẻ', 'ẽ', 'ẹ', 'e' ],
            ['ê', 'ề', 'ế', 'ể', 'ễ', 'ệ', 'ee'],
            ['i', 'ì', 'í', 'ỉ', 'ĩ', 'ị', 'i' ],
            ['o', 'ò', 'ó', 'ỏ', 'õ', 'ọ', 'o' ],
            ['ô', 'ồ', 'ố', 'ổ', 'ỗ', 'ộ', 'o'],
            ['ơ', 'ờ', 'ớ', 'ở', 'ỡ', 'ợ', 'ow'],
            ['u', 'ù', 'ú', 'ủ', 'ũ', 'ụ', 'u' ],
            ['ư', 'ừ', 'ứ', 'ử', 'ữ', 'ự', 'uw'],
            ['y', 'ỳ', 'ý', 'ỷ', 'ỹ', 'ỵ', 'y' ]
        ]
        pass

    def createVowelsTable(self):
        """Create Vowels Table"""
        for i in range(len(self.vowels_table)):
            for j in range(len(self.vowels_table[i]) - 1):
                self.vowels_to_ids[self.vowels_table[i][j]] = (i, j)

    def IsValidVietnameseWord(self,word):
        """Nguyên âm chỉ có thể đứng chung với nguyên âm. Một từ không thể có 2 nguyên âm cách nhau bởi 1 phụ âm"""
        chars = list(word)
        #nguyen am
        vowel_index = -1
        for i in range(len(chars)):
            idx_vowel_table = self.vowels_to_ids.get(chars[i],(-1,-1))[0]
            if idx_vowel_table != -1:
                if vowel_index == -1:
                    vowel_index = i
                else:
                    if i - vowel_index != 1:
                        return False
                    vowel_index = i
        return True

    def WordStandardized(self,word):
        """Standardize Word"""
        if not self.IsValidVietnameseWord(word):
            return word

        chars = list(word)
        vowel_indexes = []

        # tìm vị trí nguyên âm
        qu_or_gi = False
        thanh_dieu = 0
        for i in range(len(chars)):
            vowel_table_row, vowel_table_col = self.vowels_to_ids.get(chars[i],(-1,-1))
            if vowel_table_row == -1 :
                continue
            # qu
            if vowel_table_row == 9:
                if i != 0 and chars[i-1] == 'q':
                    chars[i] = 'u'
                    qu_or_gi = True
            # gi
            elif vowel_table_row == 5:
                if i != 0 and chars[i-1] == 'g':
                    chars[i] = 'i'
                    qu_or_gi = True

            # có chứa thanh điệu
            if vowel_table_col != 0:
                thanh_dieu = vowel_table_col
                chars[i] = self.vowels_table[vowel_table_row][0]

            vowel_indexes.append(i)
        # 1 nguyên âm
        if len(vowel_indexes) == 1:
            c = chars[vowel_indexes[0]]
            chars[vowel_indexes[0]] = self.vowels_table[self.vowels_to_ids[c][0]][thanh_dieu]
            return ''.join(chars)

        for idx_vowel in vowel_indexes:
            vowel_table_row, vowel_table_col = self.vowels_to_ids.get(chars[idx_vowel],(-1,-1))
            #ê, ơ, ô
            if vowel_table_row == 4 or vowel_table_row == 7 or vowel_table_row == 8:
                c = chars[idx_vowel]
                chars[idx_vowel] = self.vowels_table[self.vowels_to_ids[c][0]][thanh_dieu]
                return ''.join(chars)

            # kiểm tra qu và gi, 2-3 nguyên âm thì nguyên âm thứ 2 chứa dấu
            if qu_or_gi:
                if len(vowel_indexes) == 2 or len(vowel_indexes) == 3:
                    c = chars[vowel_indexes[1]]
                    chars[vowel_indexes[1]] = self.vowels_table[self.vowels_to_ids[c][0]][thanh_dieu]
                return ''.join(chars)

            # 2 nguyên âm
            if len(vowel_indexes) == 2:
                # âm cuối là nguyên âm
                if vowel_indexes[-1] == len(chars) - 1:
                    c = chars[vowel_indexes[0]]
                    chars[vowel_indexes[0]] = self.vowels_table[self.vowels_to_ids[c][0]][thanh_dieu]
                else:
                    c = chars[vowel_indexes[-1]]
                    chars[vowel_indexes[-1]] = self.vowels_table[self.vowels_to_ids[c][0]][thanh_dieu]
                return ''.join(chars)

            elif len(vowel_indexes) == 3:
                # âm cuối là nguyên âm
                if vowel_indexes[-1] == len(chars) - 1:
                    c = chars[vowel_indexes[1]]
                    chars[vowel_indexes[1]] = self.vowels_table[self.vowels_to_ids[c][0]][thanh_dieu]
                else:
                    c = chars[vowel_indexes[-1]]
                    chars[vowel_indexes[-1]] = self.vowels_table[self.vowels_to_ids[c][0]][thanh_dieu]
                return ''.join(chars)

        return ''.join(chars)

    def normalize(self,text):

        #Chuyen sang viet thuong
        text = text.lower()

        # Rút gọn từ kéo dài
        text = re.sub(r'(\w)\1+',r'\1',text)

        # xóa các emoji dư thừa
        emoji_pattern = re.compile(
            "["
            "\U0001F600-\U0001F64F"  # Emoticons
            "\U0001F300-\U0001F5FF"  # Symbols & pictographs
            "\U0001F680-\U0001F6FF"  # Transport & map symbols
            "\U0001F1E0-\U0001F1FF"  # Flags (iOS)
            "\U00002500-\U00002BEF"  # Chinese char
            "\U00002702-\U000027B0"
            "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251"
            "\U0001f926-\U0001f937"
            "\U00010000-\U0010ffff"
            "]+",
            flags=re.UNICODE,
        )
        text = emoji_pattern.sub(r'',text) # no emoji

        # xóa space d
        text = re.sub(r"( )\1+",r'\1',text)
        text = re.sub(r"[:)^@!`~%;?(\+\-\'\"]+",r'',text)
        text = text.replace("“","")
        # remove hastag
        text = re.sub("(@[A-Za-z0-9]+)|(#[0-9A-Za-z]+)","", text)
        return text

In [11]:
train_data = pd.read_json('/kaggle/input/macsaa/train.json')
dev_data = pd.read_json('/kaggle/input/macsaa/dev.json')
test_data = pd.read_json('/kaggle/input/macsaa/test.json')
# data['image_label'].apply(lambda x: len(x)).value_counts()

FileNotFoundError: File /kaggle/input/macsaa/train.json does not exist

In [ ]:
normalize_class = TextNormalize()

In [ ]:
train_data['comment'] = train_data['comment'].apply(lambda x: normalize_class.normalize(text_normalize(convert_unicode(x))))
dev_data['comment'] = dev_data['comment'].apply(lambda x: normalize_class.normalize(text_normalize(convert_unicode(x))))
test_data['comment'] = test_data['comment'].apply(lambda x: normalize_class.normalize(text_normalize(convert_unicode(x))))

In [ ]:
train_data['comment'].values[2]

In [ ]:
train_data

In [ ]:
ASPECT = ['Location', 'Food', 'Room', 'Facilities', 'Service', 'Public_area']

In [ ]:
train_data.head()

In [ ]:
roi_df = pd.read_csv("/kaggle/input/macsaa/roi_data.csv")
roi_df['file_name'] = roi_df['file_name'] + '.png'

In [ ]:
import json 

with open('/kaggle/input/macsaa/resnet152_image_label.json') as imf:
    dict_image_aspect = json.load(imf)

with open('/kaggle/input/macsaa/resnet152_roi_label.json') as rf:
    dict_roi_aspect = json.load(rf)


In [ ]:
from torchvision.models import resnet152,ResNet152_Weights,ResNet50_Weights
import torch.nn.functional as F
from torchvision.io import read_image, ImageReadMode

#by default: num_img = 7
#            num_roi = 4
# return:
#   [CLS] aspect category [SEP] text [SEP] list image aspect (split by , ) [SEP] list roi aspect (split by , ) [SEP] 
#   image: (num_img, 3, 224, 224)
#   roi: (num_img, num_roi, 3, 224, 224) 

class MACSADataset(torch.utils.data.Dataset):
    def __init__(self, data,pretrained_path, img_folder, roi_df, dict_image_aspect,dict_roi_aspect, num_img, num_roi):
        self.data = data
        self.ASPECT = ['Location', 'Food', 'Room', 'Facilities', 'Service', 'Public_area']

        self.pola_to_num = {
            "None": 0,
            "Negative": 1,
            "Neutral": 2,
            "Positive": 3
        }

        self.transform = transforms.Compose([
                            transforms.Resize((224,224),antialias=True),  # args.crop_size, by default it is set to be 224
                            transforms.ConvertImageDtype(torch.float32),
                            transforms.Normalize((0.485, 0.456, 0.406),
                                                    (0.229, 0.224, 0.225))
                                        ])
        self.roi_df = roi_df
        self.img_folder = img_folder
        self.dict_image_aspect = dict_image_aspect
        self.dict_roi_aspect = dict_roi_aspect
        self.num_img = num_img
        self.num_roi = num_roi
        self.tokenizer = AutoTokenizer.from_pretrained(pretrained_path)

    def __len__(self):
        return self.data.shape[0]
    def __getitem__(self,idx):

        idx_data = self.data.iloc[idx,:].values

        text = idx_data[0]
        
        list_image_aspect = []
        list_roi_aspect = []
        for img_name in idx_data[1]:
            try:
                list_image_aspect.extend(self.dict_image_aspect[img_name])
            except:
                pass
            try:
                list_roi_aspect.extend(self.dict_roi_aspect[img_name])
            except:
                pass
        list_image_aspect = list(set(list_image_aspect))
        list_roi_aspect = list(set(list_roi_aspect))

        if len(list_image_aspect) == 0:
            list_image_aspect = ['empty']
        if len(list_roi_aspect) == 0:
            list_roi_aspect = ['empty']

        text_img_label = idx_data[3]
        list_aspect, list_polar = [],[]
        for asp_pol in text_img_label:
            asp,pol = asp_pol.split("#")
            if "_" in asp:
                asp = "Public area"
            list_aspect.append(asp)
            list_polar.append(pol)

        for asp in self.ASPECT:
            if "_" in asp:
                asp = "Public area"
            if asp not in list_aspect:
                list_aspect.append(asp)
                list_polar.append('None')
        
        all_input_ids = []
        all_token_types_ids = []
        all_attn_mask = []
        all_label_id = []
        all_added_input_mask = []
        all_s2_input_ids = []
        all_s2_token_type_ids = []
        all_s2_attention_mask = []

        for ix in range(len(self.ASPECT)):
            asp = self.ASPECT[ix]
            if "_" in asp:
                asp = "Public area"

            idx_asp_in_list_asp = list_aspect.index(asp)

            joined_aspect = f" {' , '.join(list_image_aspect)} </s></s>  {' , '.join(list_roi_aspect)}"
            joined_aspect = joined_aspect.lower().replace('_',' ')

            combine_text = f"{asp} </s></s> {text}"
            combine_text = combine_text.lower().replace('_',' ')
            tokens = self.tokenizer(combine_text, joined_aspect, max_length=170,truncation='only_first',padding='max_length', return_token_type_ids=True)

            asp_token = self.tokenizer(asp,max_length=6,truncation=True,padding='max_length', return_token_type_ids=True)
            s2_input_ids = torch.tensor(asp_token['input_ids'])
            s2_token_type_ids = torch.tensor(asp_token['token_type_ids'])
            s2_attention_mask = torch.tensor(asp_token['attention_mask'])

            input_ids = torch.tensor(tokens['input_ids'])
            token_type_ids = torch.tensor(tokens['token_type_ids'])
            attention_mask = torch.tensor(tokens['attention_mask'])
            added_input_mask =torch.tensor( [1] * (170+49))

            label_id = list_polar[idx_asp_in_list_asp]

            all_input_ids.append(input_ids)
            all_token_types_ids.append(token_type_ids)
            all_attn_mask.append(attention_mask)
            all_added_input_mask.append(added_input_mask)
            all_label_id.append(self.pola_to_num[label_id])
            all_s2_input_ids.append(s2_input_ids)
            all_s2_token_type_ids.append(s2_token_type_ids)
            all_s2_attention_mask.append(s2_attention_mask)

        all_input_ids = torch.stack(all_input_ids,dim=0)
        all_token_types_ids = torch.stack(all_token_types_ids)
        all_attn_mask = torch.stack(all_attn_mask)
        all_added_input_mask = torch.stack(all_added_input_mask)
        all_s2_input_ids = torch.stack(all_s2_input_ids)
        all_s2_token_type_ids = torch.stack(all_s2_token_type_ids)
        all_s2_attention_mask = torch.stack(all_s2_attention_mask)

        all_label_id = torch.tensor(all_label_id)

        list_img_path = idx_data[1]
        
        list_img_features = []
        global_roi_features = [] # num_img, num_roi, 3, 224, 224
        global_roi_coor = []
        for img_path in list_img_path:
          image_os_path = os.path.join(self.img_folder, img_path)
          one_image = read_image(image_os_path, mode = ImageReadMode.RGB)
          img_transform = self.transform(one_image).unsqueeze(0) # 1, 3, 224, 224
          list_img_features.append(img_transform)

          ##### ROI
          list_roi_img = [] # num_roi, 3, 224, 224
          list_roi_coor = [] # num_roi, 4
          roi_in_img_df = self.roi_df[self.roi_df['file_name'] == img_path][:self.num_roi]
          if roi_in_img_df.shape[0] == 0:
              list_roi_img = np.zeros((self.num_roi,3,224,224))
              
              global_roi_coor.append(np.zeros((self.num_roi,4)))
              global_roi_features.append(list_roi_img)
              continue
          
          for i_roi in range(roi_in_img_df.shape[0]):
            x1, x2, y1, y2 = roi_in_img_df.iloc[i_roi,1:4+1].values            

            roi_in_image = one_image[:,x1:x2,y1:y2]
            roi_transform = self.transform(roi_in_image).numpy() # 3, 224, 224

            x1, x2, y1, y2 = x1/512, x2/512, y1/512, y2/512
            cv = lambda x: np.clip([x],0.0,1.0)[0]
            x1 = cv(x1)
            x2 = cv(x2)
            y1 = cv(y1)
            y2 = cv(y2) 

            list_roi_coor.append([x1,x2,y1,y2])
            list_roi_img.append(roi_transform)

          if i_roi < self.num_roi:
            for k in range(self.num_roi - i_roi-1):
                list_roi_img.append(np.zeros((3,224,224)))
                list_roi_coor.append(np.zeros((4,)))

          global_roi_features.append(list_roi_img)
          global_roi_coor.append(list_roi_coor)

        ### FOR IMAGE
        t_img_features = torch.zeros((self.num_img, 3, 224, 224))
        num_imgs = len(list_img_features)

        if num_imgs >= self.num_img:
            for i in range(self.num_img):
                t_img_features[i,:] = list_img_features[i]
        else:
            for i in range(self.num_img):
                if i < num_imgs:
                    t_img_features[i, :] = list_img_features[i]
                else:
                    break
        
        ### FOR ROI
        global_roi_features = np.asarray(global_roi_features)
        global_roi_coor = np.asarray(global_roi_coor)

        roi_img_features = np.zeros((self.num_img, self.num_roi, 3, 224, 224))
        roi_coors = np.zeros((self.num_img,self.num_roi,4))

        num_img_roi = len(global_roi_features)

        if num_img_roi >= self.num_img:
            for i in range(self.num_img):
                roi_img_features[i,:] = global_roi_features[i]
                roi_coors[i,:] = global_roi_coor[i]
        else:
            for i in range(self.num_img):
                if i < num_imgs:
                    roi_img_features[i, :] = global_roi_features[i]
                    roi_coors[i,:] = global_roi_coor[i,:]
                else:
                    break

        roi_img_features = torch.tensor(roi_img_features)
        roi_coors = torch.tensor(roi_coors)

        return t_img_features, roi_img_features, roi_coors, all_input_ids, all_token_types_ids, all_attn_mask,all_s2_input_ids, all_s2_token_type_ids, all_s2_attention_mask, all_added_input_mask, all_label_id


In [ ]:
PRETRAINED_PATH = "xlm-roberta-base"

In [ ]:
NUM_ROIS = 4
NUM_IMGS = 7

In [ ]:
train_set = MACSADataset(train_data,PRETRAINED_PATH,"/kaggle/input/macsaa/image/image",roi_df, dict_image_aspect, dict_roi_aspect, NUM_IMGS, NUM_ROIS)
dev_set = MACSADataset(dev_data,PRETRAINED_PATH,"/kaggle/input/macsaa/image/image",roi_df, dict_image_aspect, dict_roi_aspect, NUM_IMGS, NUM_ROIS)
test_set = MACSADataset(test_data,PRETRAINED_PATH,"/kaggle/input/macsaa/image/image",roi_df, dict_image_aspect, dict_roi_aspect, NUM_IMGS, NUM_ROIS)

In [ ]:
z = 2
roi_s = train_set[z][1] # num_img, num_roi, 3, 224, 224
roi_s[1][0]

In [ ]:
z = 2
roi_coor = train_set[z][2] # num_img, num_roi, 4
print(roi_coor.size())
roi_coor[1]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_PATH)
aa = 181
for iz in range(6):
    print(train_set[aa][-1][iz])
    for i in train_set[aa][3][iz]:
        print(tokenizer.convert_ids_to_tokens(i.item()),end=' ' )
    print()

In [ ]:
train_loader = torch.utils.data.DataLoader(train_set, batch_size = 4, shuffle=True)
dev_loader = torch.utils.data.DataLoader(dev_set, batch_size = 4, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_set, batch_size = 4, shuffle=False)

In [ ]:
for i in train_loader:
  t_img_features, roi_img_features, roi_coor, all_input_ids, all_token_types_ids, all_attn_mask,all_s2_input_ids, all_s2_token_type_ids, all_s2_attention_mask, all_added_input_mask, all_label_id = i
  # print(img.size())
  break
  

In [ ]:
roi_img_features.size()

In [ ]:
train_loader

# MODEL

In [ ]:
import torch
import torch.nn as nn
import copy
import numpy as np
import math
import torch.nn.functional as F
from transformers import AutoModel
from torch.autograd import Variable


In [ ]:
import os
mode = 'FP16'
os.environ["TORCH_CUDNN_V8_API_ENABLED"] = "1"
torch.backends.cuda.matmul.allow_tf32 = True if mode == 'TF32' else False
scaler = torch.cuda.amp.GradScaler(enabled=True if mode == 'FP16' else False)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class FeatureExtractor(torch.nn.Module):
  def __init__(self, pretrained_path):
    super(FeatureExtractor,self).__init__()
    self.cell = AutoModel.from_pretrained(pretrained_path)

  def forward(self, input_ids, token_type_ids, attention_mask):
    seq_out, pooled_out = self.cell(input_ids = input_ids,
                                    token_type_ids = token_type_ids, 
                                    attention_mask = attention_mask )[:2]

    return seq_out, pooled_out


In [ ]:
def gelu(x):
    """Implementation of the gelu activation function.
        For information: OpenAI GPT's gelu is slightly different (and gives slightly different results):
        0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))
    """
    return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

def swish(x):
    return x * torch.sigmoid(x)


## Attention module

In [ ]:
HIDDEN_SIZE=768
NUM_HIDDEN_LAYERS=12
NUM_ATTENTION_HEADS=12 
INTERMEDIATE_SIZE=3072
HIDDEN_ACT="gelu"
HIDDEN_DROPOUT_PROB=0.1
ATTENTION_PROBS_DROPOUT_PROB=0.1
MAX_POSITION_EMBEDDINGS=512
TYPE_VOCAB_SIZE=2
INITIALIZER_RANGE=0.02

ACT2FN = {"gelu": gelu, "relu": torch.nn.functional.relu, "swish": swish}


### Attention

In [ ]:
# for NAACL visual attention part
class Attention(nn.Module):
    def __init__(self, embed_dim, hidden_dim=None, n_head=1, score_function='scaled_dot_product', dropout=0.1):
        ''' Attention Mechanism
        :param embed_dim:
        :param hidden_dim:
        :param n_head: num of head (Multi-Head Attention)
        :param score_function: scaled_dot_product / mlp (concat) / bi_linear (general dot)
        '''
        super(Attention, self).__init__()
        if hidden_dim is None:
            hidden_dim = embed_dim // n_head
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.n_head = n_head
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.score_function = score_function
        self.w_kx = nn.Parameter(torch.FloatTensor(n_head, embed_dim, hidden_dim))
        self.w_qx = nn.Parameter(torch.FloatTensor(n_head, embed_dim, hidden_dim))
        self.proj = nn.Linear(n_head * hidden_dim, embed_dim)
        if score_function == 'mlp':
            self.weight = nn.Parameter(torch.Tensor(hidden_dim * 2, 1))
        elif self.score_function == 'bi_linear':
            self.weight = nn.Parameter(torch.Tensor(hidden_dim, hidden_dim))
        else:
            self.register_parameter('weight', None)

    def forward(self, k, q, memory_len):
        if len(q.shape) == 2:  # q_len missing
            q = torch.unsqueeze(q, dim=1)
        if len(k.shape) == 2:  # k_len missing
            k = torch.unsqueeze(k, dim=1)
        mb_size = k.shape[0]  # ?
        k_len = k.shape[1]
        q_len = q.shape[1]
        # k: (?, k_len, embed_dim,)
        # q: (?, q_len, embed_dim,)
        # kx: (n_head, ?*k_len, embed_dim) -> (n_head*?, k_len, hidden_dim)
        # qx: (n_head, ?*q_len, embed_dim) -> (n_head*?, q_len, hidden_dim)
        # score: (n_head*?, q_len, k_len,)
        # output: (?, q_len, embed_dim,)
        kx = k.repeat(self.n_head, 1, 1).view(self.n_head, -1, self.embed_dim)  # (n_head, ?*k_len, embed_dim)
        qx = q.repeat(self.n_head, 1, 1).view(self.n_head, -1, self.embed_dim)  # (n_head, ?*q_len, embed_dim)
        kx = torch.bmm(kx, self.w_kx).view(-1, k_len, self.hidden_dim)  # (n_head*?, k_len, hidden_dim)
        qx = torch.bmm(qx, self.w_qx).view(-1, q_len, self.hidden_dim)  # (n_head*?, q_len, hidden_dim)
        if self.score_function == 'scaled_dot_product':
            kt = kx.permute(0, 2, 1)
            qkt = torch.bmm(qx, kt)
            score = torch.div(qkt, math.sqrt(self.hidden_dim))
        elif self.score_function == 'mlp':
            kxx = torch.unsqueeze(kx, dim=1).expand(-1, q_len, -1, -1)
            qxx = torch.unsqueeze(qx, dim=2).expand(-1, -1, k_len, -1)
            kq = torch.cat((kxx, qxx), dim=-1)  # (n_head*?, q_len, k_len, hidden_dim*2)
            score = torch.tanh(torch.matmul(kq, self.weight).squeeze(dim=-1))
        elif self.score_function == 'bi_linear':
            qw = torch.matmul(qx, self.weight)
            kt = kx.permute(0, 2, 1)
            score = torch.tanh(torch.bmm(qw, kt))
        else:
            raise RuntimeError('invalid score_function')
        score = F.softmax(score, dim=-1)
        attentions = torch.squeeze(score, dim=1)
        # print(attentions[:2])
        # create mask based on the sentence lengths
        mask = Variable(torch.ones(attentions.size())).to(self.device)
        for i, l in enumerate(memory_len):
            if l < k_len:
                mask[i, l:] = 0
        # apply mask and renormalize attention scores (weights)
        masked = attentions * mask
        # print(masked[:2])
        # print(masked.shape)
        _sums = masked.sum(-1)  # sums per row
        attentions = torch.div(masked, _sums.view(_sums.size(0), 1))
        # print(attentions[:2])

        score = torch.unsqueeze(attentions, dim=1)

        output = torch.bmm(score, kx)  # (n_head*?, k_len, hidden_dim)
        output = torch.cat(torch.split(output, mb_size, dim=0), dim=-1)  # (?, k_len, n_head*hidden_dim)
        output = self.proj(output)  # (?, k_len, embed_dim)
        return output

### BERT MID 

In [ ]:
class BertLayerNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-12):
        """Construct a layernorm module in the TF style (epsilon inside the square root).
        """
        super(BertLayerNorm, self).__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.bias = nn.Parameter(torch.zeros(hidden_size))
        self.variance_epsilon = eps

    def forward(self, x):
        u = x.mean(-1, keepdim=True)
        s = (x - u).pow(2).mean(-1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.variance_epsilon)
        return self.weight * x + self.bias

In [ ]:
class BertIntermediate(nn.Module):
    def __init__(self):
        super(BertIntermediate, self).__init__()
        self.dense = nn.Linear(HIDDEN_SIZE, INTERMEDIATE_SIZE)
        self.intermediate_act_fn = ACT2FN[HIDDEN_ACT]

    def forward(self, hidden_states):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states

In [ ]:
class BertOutput(nn.Module):
    def __init__(self):
        super(BertOutput, self).__init__()
        self.dense = nn.Linear(INTERMEDIATE_SIZE, HIDDEN_SIZE)
        self.LayerNorm = BertLayerNorm(HIDDEN_SIZE, eps=1e-12)
        self.dropout = nn.Dropout(HIDDEN_DROPOUT_PROB)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


### Self Attention

In [ ]:
class BertSelfAttention(nn.Module):
    def __init__(self):
        super(BertSelfAttention, self).__init__()

        self.num_attention_heads = NUM_ATTENTION_HEADS
        self.attention_head_size = int(HIDDEN_SIZE / NUM_ATTENTION_HEADS)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(HIDDEN_SIZE, self.all_head_size)
        self.key = nn.Linear(HIDDEN_SIZE, self.all_head_size)
        self.value = nn.Linear(HIDDEN_SIZE, self.all_head_size)

        self.dropout = nn.Dropout(ATTENTION_PROBS_DROPOUT_PROB)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states, attention_mask):
        mixed_query_layer = self.query(hidden_states)
        mixed_key_layer = self.key(hidden_states)
        mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        # Take the dot product between "query" and "key" to get the raw attention scores.
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        # Apply the attention mask is (precomputed for all layers in BertModel forward() function)
        attention_scores = attention_scores + attention_mask

        # Normalize the attention scores to probabilities.
        attention_probs = nn.Softmax(dim=-1)(attention_scores)

        # This is actually dropping out entire tokens to attend to, which might
        # seem a bit unusual, but is taken from the original Transformer paper.
        attention_probs = self.dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        return context_layer


In [ ]:
class BertSelfOutput(nn.Module):
    def __init__(self):
        super(BertSelfOutput, self).__init__()
        self.dense = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.LayerNorm = BertLayerNorm(HIDDEN_SIZE, eps=1e-12)
        self.dropout = nn.Dropout(HIDDEN_DROPOUT_PROB)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


In [ ]:
class BertAttention(nn.Module):
    def __init__(self):
        super(BertAttention, self).__init__()
        self.self = BertSelfAttention()
        self.output = BertSelfOutput()

    def forward(self, input_tensor, attention_mask):
        self_output = self.self(input_tensor, attention_mask)
        attention_output = self.output(self_output, input_tensor)
        return attention_output


### Cross Attention

In [ ]:
class BertCoAttention(nn.Module):
    def __init__(self):
        super(BertCoAttention, self).__init__()

        self.num_attention_heads = NUM_ATTENTION_HEADS
        self.attention_head_size = int(HIDDEN_SIZE / NUM_ATTENTION_HEADS)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(HIDDEN_SIZE, self.all_head_size)
        self.key = nn.Linear(HIDDEN_SIZE, self.all_head_size)
        self.value = nn.Linear(HIDDEN_SIZE, self.all_head_size)

        self.dropout = nn.Dropout(ATTENTION_PROBS_DROPOUT_PROB)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, s1_hidden_states, s2_hidden_states, s2_attention_mask):
        mixed_query_layer = self.query(s1_hidden_states)
        mixed_key_layer = self.key(s2_hidden_states)
        mixed_value_layer = self.value(s2_hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        # Take the dot product between "query" and "key" to get the raw attention scores.
        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        # Apply the attention mask is (precomputed for all layers in BertModel forward() function)
        attention_scores = attention_scores + s2_attention_mask

        # Normalize the attention scores to probabilities.
        attention_probs = nn.Softmax(dim=-1)(attention_scores)

        # This is actually dropping out entire tokens to attend to, which might
        # seem a bit unusual, but is taken from the original Transformer paper.
        attention_probs = self.dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)
        return context_layer

In [ ]:
class BertCrossAttention(nn.Module):
    def __init__(self):
        super(BertCrossAttention, self).__init__()
        self.self = BertCoAttention()
        self.output = BertSelfOutput()

    def forward(self, s1_input_tensor, s2_input_tensor, s2_attention_mask):
        s1_cross_output = self.self(s1_input_tensor, s2_input_tensor, s2_attention_mask)
        attention_output = self.output(s1_cross_output, s1_input_tensor)
        return attention_output


### Multimodal Attention

In [ ]:
class BertLayer(nn.Module):
    def __init__(self):
        super(BertLayer, self).__init__()
        self.attention = BertAttention()
        self.intermediate = BertIntermediate()
        self.output = BertOutput()

    def forward(self, hidden_states, attention_mask):
        attention_output = self.attention(hidden_states, attention_mask)
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output

In [ ]:
class MultimodalEncoder(nn.Module):
    def __init__(self):
        super(MultimodalEncoder, self).__init__()
        layer = BertLayer()
        self.layer = nn.ModuleList([copy.deepcopy(layer) for _ in range(1)])

    def forward(self, hidden_states, attention_mask, output_all_encoded_layers=True):
        all_encoder_layers = []
        for layer_module in self.layer:
            hidden_states = layer_module(hidden_states, attention_mask)
            if output_all_encoded_layers:
                all_encoder_layers.append(hidden_states)
        if not output_all_encoded_layers:
            all_encoder_layers.append(hidden_states)
        return all_encoder_layers

### Crossmodal Attention

In [ ]:
class BertCrossAttentionLayer(nn.Module):
    def __init__(self):
        super(BertCrossAttentionLayer, self).__init__()
        self.attention = BertCrossAttention()
        self.intermediate = BertIntermediate()
        self.output = BertOutput()

    def forward(self, s1_hidden_states, s2_hidden_states, s2_attention_mask):
        attention_output = self.attention(s1_hidden_states, s2_hidden_states, s2_attention_mask)
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output


class BertCrossEncoder(nn.Module):
    def __init__(self):
        super(BertCrossEncoder, self).__init__()
        layer = BertCrossAttentionLayer()
        self.layer = nn.ModuleList([copy.deepcopy(layer) for _ in range(1)])

    def forward(self, s1_hidden_states, s2_hidden_states, s2_attention_mask, output_all_encoded_layers=True):
        all_encoder_layers = []
        for layer_module in self.layer:
            s1_hidden_states = layer_module(s1_hidden_states, s2_hidden_states, s2_attention_mask)
            if output_all_encoded_layers:
                all_encoder_layers.append(s1_hidden_states)
        if not output_all_encoded_layers:
            all_encoder_layers.append(s1_hidden_states)
        return all_encoder_layers

### POOLING

In [ ]:
class BertPooler(nn.Module):
    def __init__(self):
        super(BertPooler, self).__init__()
        self.dense = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.activation = nn.Tanh()

    def forward(self, hidden_states):
        # We "pool" the model by simply taking the hidden state corresponding
        # to the first token.
        first_token_tensor = hidden_states[:, 0]
        pooled_output = self.dense(first_token_tensor)
        pooled_output = self.activation(pooled_output)
        return pooled_output


## RoI module

In [ ]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [ ]:
def box_attention(query, key, value, box_relation_embds_matrix, mask=None, dropout=None):
    '''
    Compute 'Scaled Dot Product Attention as in paper Relation Networks for Object Detection'.
    Follow the implementation in https://github.com/heefe92/Relation_Networks-pytorch/blob/master/model.py#L1026-L1055
    '''

    N = value.size()[:2]
    dim_k = key.size(-1)
    dim_g = box_relation_embds_matrix.size()[-1]

    w_q = query
    w_k = key.transpose(-2, -1)
    w_v = value

    #attention weights
    scaled_dot = torch.matmul(w_q,w_k)
    scaled_dot = scaled_dot / np.sqrt(dim_k)
    if mask is not None:
        scaled_dot = scaled_dot.masked_fill(mask == 0, -1e9)

    #w_g = box_relation_embds_matrix.view(N,N)
    w_g = box_relation_embds_matrix
    w_a = scaled_dot
    #w_a = scaled_dot.view(N,N)

    # multiplying log of geometric weights by feature weights
    w_mn = torch.log(torch.clamp(w_g, min = 1e-6)) + w_a
    w_mn = torch.nn.Softmax(dim=-1)(w_mn)
    if dropout is not None:
        w_mn = dropout(w_mn)

    output = torch.matmul(w_mn,w_v)

    return output, w_mn

In [ ]:
class BoxMultiHeadedAttention(nn.Module):
    '''
    Self-attention layer with relative position weights.
    Following the paper "Relation Networks for Object Detection" in https://arxiv.org/pdf/1711.11575.pdf
    '''

    def __init__(self, h, d_model, trignometric_embedding=True, legacy_extra_skip=False, dropout=0.1):
        "Take in model size and number of heads."
        super(BoxMultiHeadedAttention, self).__init__()

        assert d_model % h == 0
        self.trignometric_embedding=trignometric_embedding
        self.legacy_extra_skip = legacy_extra_skip

        # We assume d_v always equals d_k
        self.h = h
        self.d_k = d_model // h
        if self.trignometric_embedding:
            self.dim_g = 64
        else:
            self.dim_g = 4
        geo_feature_dim = self.dim_g

        #matrices W_q, W_k, W_v, and one last projection layer
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.WGs = clones(nn.Linear(geo_feature_dim, 1, bias=True),8)

        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def BoxRelationalEmbedding(self,f_g, dim_g=64, wave_len=1000, trignometric_embedding= True):
        """
        Given a tensor with bbox coordinates for detected objects on each batch image,
        this function computes a matrix for each image

        with entry (i,j) given by a vector representation of the
        displacement between the coordinates of bbox_i, and bbox_j

        input: np.array of shape=(batch_size, max_nr_bounding_boxes, 4)
        output: np.array of shape=(batch_size, max_nr_bounding_boxes, max_nr_bounding_boxes, 64)
        """
        #returns a relational embedding for each pair of bboxes, with dimension = dim_g
        #follow implementation of https://github.com/heefe92/Relation_Networks-pytorch/blob/master/model.py#L1014-L1055

        batch_size = f_g.size(0)

        x_min, x_max, y_min, y_max = torch.chunk(f_g, 4, dim=-1)

        cx = (x_min + x_max) * 0.5
        cy = (y_min + y_max) * 0.5
        w = (x_max - x_min) + 1.
        h = (y_max - y_min) + 1.

        #cx.view(1,-1) transposes the vector cx, and so dim(delta_x) = (dim(cx), dim(cx))
        delta_x = cx - cx.view(batch_size, 1, -1)
        delta_x = torch.clamp(torch.abs(delta_x / w), min=1e-3)
        delta_x = torch.log(delta_x)

        delta_y = cy - cy.view(batch_size, 1, -1)
        delta_y = torch.clamp(torch.abs(delta_y / h), min=1e-3)
        delta_y = torch.log(delta_y)

        delta_w = torch.log(w / w.view(batch_size, 1, -1))
        delta_h = torch.log(h / h.view(batch_size, 1, -1))

        matrix_size = delta_h.size()
        delta_x = delta_x.view(batch_size, matrix_size[1], matrix_size[2], 1)
        delta_y = delta_y.view(batch_size, matrix_size[1], matrix_size[2], 1)
        delta_w = delta_w.view(batch_size, matrix_size[1], matrix_size[2], 1)
        delta_h = delta_h.view(batch_size, matrix_size[1], matrix_size[2], 1)

        position_mat = torch.cat((delta_x, delta_y, delta_w, delta_h), -1)

        if trignometric_embedding == True:
            feat_range = torch.arange(dim_g / 8).to('cuda')
            dim_mat = feat_range / (dim_g / 8)
            dim_mat = 1. / (torch.pow(wave_len, dim_mat))

            dim_mat = dim_mat.view(1, 1, 1, -1)
            position_mat = position_mat.view(batch_size, matrix_size[1], matrix_size[2], 4, -1)
            position_mat = 100. * position_mat

            mul_mat = position_mat * dim_mat
            mul_mat = mul_mat.view(batch_size, matrix_size[1], matrix_size[2], -1)
            sin_mat = torch.sin(mul_mat)
            cos_mat = torch.cos(mul_mat)
            embedding = torch.cat((sin_mat, cos_mat), -1)
        else:
            embedding = position_mat
        return (embedding)

    def forward(self, input_query, input_key, input_value, input_box, mask=None):
        "Implements Figure 2 of Relation Network for Object Detection"
        if mask is not None:
            # Same mask applied to all h heads.
            mask = mask.unsqueeze(1)
        nbatches = input_query.size(0)

        #tensor with entries R_mn given by a hardcoded embedding of the relative position between bbox_m and bbox_n
        relative_geometry_embeddings = self.BoxRelationalEmbedding(input_box, trignometric_embedding= self.trignometric_embedding)
        flatten_relative_geometry_embeddings = relative_geometry_embeddings.view(-1,self.dim_g)
        flatten_relative_geometry_embeddings = flatten_relative_geometry_embeddings.to(dtype=next(self.parameters()).dtype) # FP16
        
        # 1) Do all the linear projections in batch from d_model => h x d_k
        query, key, value = \
            [l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
             for l, x in zip(self.linears, (input_query, input_key, input_value))]
        box_size_per_head = list(relative_geometry_embeddings.shape[:3])
        box_size_per_head.insert(1, 1)

        # self.WGs = self.WGs.to(dtype=next(self.parameters()).dtype)
        
        relative_geometry_weights_per_head = [l(flatten_relative_geometry_embeddings).view(box_size_per_head) for l in self.WGs]
        relative_geometry_weights = torch.cat((relative_geometry_weights_per_head),1)
        relative_geometry_weights = F.relu(relative_geometry_weights)

        # 2) Apply attention on all the projected vectors in batch.
        x, self.box_attn = box_attention(query, key, value, relative_geometry_weights, mask=mask,
                                 dropout=self.dropout)

        # 3) "Concat" using a view and apply a final linear.
        x = x.transpose(1, 2).contiguous() \
             .view(nbatches, -1, self.h * self.d_k)

        # An extra internal skip connection is added. This is only
        # kept here for compatibility with some legacy models. In
        # general, there is no advantage in using it, as there is
        # already an outer skip connection surrounding this layer.
        if self.legacy_extra_skip:
            x = input_value + x

        return self.linears[-1](x)

## Fine-grained cross-modal fusion

In [ ]:
class FCMF(nn.Module):
    def __init__(self, pretrained_path, num_labels=4, num_imgs = 7, num_roi = 7):
        super(FCMF, self).__init__()
        self.num_labels = num_labels
        self.num_imgs = num_imgs
        self.num_roi = num_roi
        self.bert = FeatureExtractor(pretrained_path)
        self.dropout = nn.Dropout(HIDDEN_DROPOUT_PROB)
        self.vismap2text = nn.Linear(2048, HIDDEN_SIZE)
        self.roimap2text = nn.Linear(2048, HIDDEN_SIZE)
        self.box_head = BoxMultiHeadedAttention(8,HIDDEN_SIZE)

        self.ent2img_attention = BertCrossEncoder()
        self.ent2img_pooler = BertPooler()
        self.ent2roi_pooler = BertPooler()

        self.comb_attention = MultimodalEncoder()
        self.text_pooler = BertPooler()
        self.classifier = nn.Linear(HIDDEN_SIZE, num_labels)

    def forward(self, input_ids, visual_embeds_att, roi_embeds_att, s2_input_ids, s2_token_type_ids, s2_attention_mask, roi_coors = None, token_type_ids=None,
                attention_mask=None, added_attention_mask=None):

        sequence_output, pooled_output = self.bert(input_ids, token_type_ids, attention_mask)

        seq_len = sequence_output.size()[1]

        # visual size: batch_size, num_img, 49,2048 
        # roi size: batch_size, num_img, num_roi, 49,2048
        list_h_i = []
        list_r_i = []

        # process list images
        for i in range(self.num_imgs):
            one_visual_embeds = visual_embeds_att[:,i,:] # each image index: batch_size, 49, 2048
            converted_vis_embed_map = self.vismap2text(one_visual_embeds)  # self.batch_size, 49, hidden_dim

            # image-guided attention
            img_mask = added_attention_mask[:,:49]
            extended_img_mask = img_mask.unsqueeze(1).unsqueeze(2)
            extended_img_mask = extended_img_mask.to(dtype=next(self.parameters()).dtype) # fp16 compatibility
            extended_img_mask = (1.0 - extended_img_mask) * -10000.0

            s2_cross_encoder = self.ent2img_attention(sequence_output, converted_vis_embed_map, extended_img_mask)
            s2_cross_output_layer = s2_cross_encoder[-1]
            s2_cross_output = self.ent2img_pooler(s2_cross_output_layer) # self.batch_size, 768
            transpose_img_embed = s2_cross_output.unsqueeze(1) # self.batch_size, 1, 768

            list_h_i.append(transpose_img_embed) 

            ### EACH IMAGES HAVE _n ROI
            comb_attention_mask = added_attention_mask[:,:seq_len + self.num_roi]
            extended_roi_mask = comb_attention_mask.unsqueeze(1).unsqueeze(2)
            extended_roi_mask = extended_roi_mask.to(dtype=next(self.parameters()).dtype)  # fp16 compatibility
            extended_roi_mask = (1.0 - extended_roi_mask) * -10000.0

            roi_at_i_img = roi_embeds_att[:,i,:] # batch_size, num_roi, 2048
            converted_roi_embed_map = self.roimap2text(roi_at_i_img) # batch_size, num_roi, hidden_dim
            
            # roi_coor: batch_size, num_img, num_roi , 4 
            relative_roi = self.box_head(converted_roi_embed_map,converted_roi_embed_map,converted_roi_embed_map,roi_coors[:,i,:]) # batch_size, num_roi, hidden_dim

            text_roi_output = torch.cat((sequence_output,relative_roi), dim=1) # batch_size, seq_len + num_roi, hidden_dim

            roi_multimodal_encoder = self.comb_attention(text_roi_output, extended_roi_mask)
            roi_att_text_output_layer = roi_multimodal_encoder[-1] # batch_size, seq_len + num_roi, hidden_dim
            roi_pooling = self.ent2roi_pooler(roi_att_text_output_layer)
            transpose_roi_embed = roi_pooling.unsqueeze(1) # self.batch_size, 1, 768

            list_r_i.append(transpose_roi_embed) 

        all_h_i_features = torch.cat(list_h_i, dim = 1) # batch_size, num_img, 768
        all_r_i_features = torch.cat(list_r_i, dim = 1) # batch_size, num_img, 768

        v_m = torch.cat((sequence_output[:,0,:].unsqueeze(1), all_h_i_features,all_r_i_features),dim=1) # batch_size, 1 + num_img + num_img, 768
    
        comb_attention_mask = added_attention_mask[:,:self.num_imgs + self.num_imgs + 1]  
        extended_attention_mask = comb_attention_mask.unsqueeze(1).unsqueeze(2)
        extended_attention_mask = extended_attention_mask.to(dtype=next(self.parameters()).dtype)  # fp16 compatibility
        extended_attention_mask = (1.0 - extended_attention_mask) * -10000.0

        final_multimodal_encoder = self.comb_attention(v_m, extended_attention_mask)
        final_multimodal_encoder = final_multimodal_encoder[-1]
        
        text_output = self.text_pooler(final_multimodal_encoder)
        pooled_output = self.dropout(text_output)

        logits = self.classifier(pooled_output)

        return logits

In [ ]:
def image_encoder(model,image):
    x = model.conv1(image)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)

    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    x = model.layer4(x)

    att = F.adaptive_avg_pool2d(x,[7,7])

    return att


In [ ]:
def roi_encoder(model,image):
    x = model.conv1(image)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)

    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    x = model.layer4(x)

    fc = x.mean(3).mean(2)

    return fc

In [ ]:
from torchvision.models import resnet152,ResNet152_Weights,resnet50, ResNet50_Weights
import torch.nn.functional as F

fcmf_model = FCMF(PRETRAINED_PATH,num_imgs=NUM_IMGS,num_roi=NUM_ROIS)
fcmf_model.to(device)

visual_model = resnet152(weights = ResNet152_Weights.IMAGENET1K_V2)
visual_model.to(device)

# TRAINING

In [ ]:
param_optimizer = list(fcmf_model.named_parameters())
no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters,
                        lr=3e-5)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
NUM_EPOCHS = 8

In [ ]:
num_warm_up_step = 0
num_training_step = len(train_loader)*NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer,num_warmup_steps=num_training_step//10, num_training_steps=num_training_step)


In [ ]:
from tqdm import tqdm

def train_one_epoch(visual_model, model,optimizer, criterion, dataloader, epoch,device,scaler):
    epoch_loss = 0

    visual_model.train()
    visual_model.zero_grad()
    model.train()

    # batch accumulation parameter
    accum_iter = 2

    with tqdm(dataloader, unit="batch") as tepoch:
        for i, data in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch}")

            t_img_features, roi_img_features, roi_coors, \
            all_input_ids, all_token_types_ids, all_attn_mask, \
            all_s2_input_ids, all_s2_token_type_ids, all_s2_attention_mask, \
            all_added_input_mask, all_label_id  = data

            roi_img_features = roi_img_features.float()
            
            roi_coors = roi_coors.to(device)
            t_img_features = t_img_features.to(device)
            roi_img_features = roi_img_features.to(device)

            all_input_ids = all_input_ids.to(device)
            all_token_types_ids = all_token_types_ids.to(device)
            all_attn_mask = all_attn_mask.to(device)

            all_s2_input_ids = all_s2_input_ids.to(device)
            all_s2_token_type_ids = all_s2_token_type_ids.to(device)
            all_s2_attention_mask = all_s2_attention_mask.to(device)

            all_added_input_mask = all_added_input_mask.to(device)
            all_label_id = all_label_id.to(device)

            with torch.no_grad():
                encoded_img = []
                encoded_roi = []

                for img_idx in range(NUM_IMGS):
                    img_features = image_encoder(visual_model,t_img_features[:,img_idx,:]).view(-1,2048,49).permute(0,2,1).squeeze(1) # batch_size, 49, 2048
                    encoded_img.append(img_features)

                    roi_f = []
                    for roi_idx in range(NUM_ROIS):
                        roi_features = roi_encoder(visual_model,roi_img_features[:,img_idx,roi_idx,:]).squeeze(1) # batch_size, 1, 2048
                        roi_f.append(roi_features)

                    roi_f = torch.stack(roi_f,dim=1)
                    encoded_roi.append(roi_f)

            encoded_img = torch.stack(encoded_img,dim=1) # batch_size, num_img, 49, 2048   
            encoded_roi = torch.stack(encoded_roi,dim=1) # batch_size, num_img, num_roi, 49,2048

            all_asp_loss = 0
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if mode == 'BF16' else torch.float16, enabled=True if '16' in mode else False):

                for id_asp in range(6):
                    logits = model(
                                input_ids = all_input_ids[:,id_asp,:], token_type_ids = all_token_types_ids[:,id_asp,:], attention_mask = all_attn_mask[:,id_asp,:], 
                                s2_input_ids = all_s2_input_ids[:,id_asp,:], s2_token_type_ids = all_s2_token_type_ids[:,id_asp,:], s2_attention_mask = all_s2_attention_mask[:,id_asp,:],
                                added_attention_mask = all_added_input_mask[:,id_asp,:],
                                visual_embeds_att = encoded_img,
                                roi_embeds_att = encoded_roi,
                                roi_coors = roi_coors
                                )
                    l = criterion(logits,all_label_id[:,id_asp])
                    all_asp_loss += l

            lll = all_asp_loss.mean().item()
            epoch_loss += lll
            
            all_asp_loss = all_asp_loss / accum_iter
            scaler.scale(all_asp_loss).backward()
            
            if ((i + 1) % accum_iter == 0) or (i + 1 == len(dataloader)):
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            tepoch.set_postfix(loss=lll)

        epoch_loss /= i
        print(f"At EPOCH {epoch}, loss = {epoch_loss}")
        print("======================================")

    return epoch_loss

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def macro_f1(y_true, y_pred):
    p_macro, r_macro, f_macro, support_macro \
      = precision_recall_fscore_support(y_true, y_pred, average='macro',zero_division=0.0,labels = [0,1,2,3])
    return p_macro, r_macro, f_macro


In [ ]:
ASPECT = [
    "Location",
    "Facilities",
    "Service",
    "Public_area",
    "Room",
    "Food"
]

def eval(visual_model, model, criterion,dataloader,device):

  print("EVAL STEP: ")

  visual_model.eval()
  model.eval()

  idx2asp = {i:v for i,v in enumerate(ASPECT)}
  epoch_loss = 0

  true_label_list = {
    "Location":[],
    "Facilities":[],
    "Service":[],
    "Public_area":[],
    "Room":[],
    "Food":[]
  }

  pred_label_list = {
    "Location":[],
    "Facilities":[],
    "Service":[],
    "Public_area":[],
    "Room":[],
    "Food":[]
  }
  with tqdm(dataloader, unit="batch") as tepoch:
    for i, data in enumerate(tepoch):

      t_img_features, roi_img_features, roi_coors, \
      all_input_ids, all_token_types_ids, all_attn_mask, \
      all_s2_input_ids, all_s2_token_type_ids, all_s2_attention_mask, \
      all_added_input_mask, all_label_id  = data
    
      roi_img_features = roi_img_features.float()

      roi_coors = roi_coors.to(device)
      t_img_features = t_img_features.to(device)
      roi_img_features = roi_img_features.to(device)

      all_input_ids = all_input_ids.to(device)
      all_token_types_ids = all_token_types_ids.to(device)
      all_attn_mask = all_attn_mask.to(device)

      all_s2_input_ids = all_s2_input_ids.to(device)
      all_s2_token_type_ids = all_s2_token_type_ids.to(device)
      all_s2_attention_mask = all_s2_attention_mask.to(device)

      all_added_input_mask = all_added_input_mask.to(device)
      all_label_id = all_label_id.to(device)
      
      with torch.no_grad():
        encoded_img = []
        encoded_roi = []

        for img_idx in range(NUM_IMGS):
            img_features = image_encoder(visual_model,t_img_features[:,img_idx,:]).view(-1,2048,49).permute(0,2,1).squeeze(1) # batch_size, 49, 2048
            encoded_img.append(img_features)

            roi_f = []
            for roi_idx in range(NUM_ROIS):
                roi_features = roi_encoder(visual_model,roi_img_features[:,img_idx,roi_idx,:]).squeeze(1) # batch_size, 1, 2048
                roi_f.append(roi_features)

            roi_f = torch.stack(roi_f,dim=1)
            encoded_roi.append(roi_f)

        encoded_img = torch.stack(encoded_img,dim=1) # batch_size, num_img, 49, 2048   
        encoded_roi = torch.stack(encoded_roi,dim=1) # batch_size, num_img, num_roi, 49,2048

      with torch.no_grad():

          all_asp_loss = 0
          for id_asp in range(6):
              logits = model(
                    input_ids = all_input_ids[:,id_asp,:], token_type_ids = all_token_types_ids[:,id_asp,:], attention_mask = all_attn_mask[:,id_asp,:], 
                    s2_input_ids = all_s2_input_ids[:,id_asp,:], s2_token_type_ids = all_s2_token_type_ids[:,id_asp,:], s2_attention_mask = all_s2_attention_mask[:,id_asp,:],
                    added_attention_mask = all_added_input_mask[:,id_asp,:],
                    visual_embeds_att = encoded_img,
                    roi_embeds_att = encoded_roi,
                    roi_coors = roi_coors
              )

              eval_loss = criterion(logits,all_label_id[:,id_asp])

              all_asp_loss += eval_loss
              asp_label = all_label_id[:,id_asp].to('cpu').numpy()
              logits = logits.detach().cpu().numpy()
              pred = np.argmax(logits,axis = -1)

              true_label_list[idx2asp[id_asp]].append(asp_label)
              pred_label_list[idx2asp[id_asp]].append(pred)

          epoch_loss += all_asp_loss.mean().item()
    epoch_loss /= i
    print(f"Loss = {epoch_loss}")

    all_precision, all_recall, all_f1 = 0, 0, 0
    for id_asp in range(6):
      tr = np.concatenate(true_label_list[idx2asp[id_asp]])
      pr = np.concatenate(pred_label_list[idx2asp[id_asp]])
      # print(tr)
      # print(pr)

      precision, recall, f1_score = macro_f1(tr,pr)
      print(f"{idx2asp[id_asp]}: ")
      print("- Precision:", precision)
      print("- Recall:", recall)
      print("- F1-score:", f1_score)
      print("========================")

      all_precision += precision
      all_recall += recall
      all_f1 += f1_score

  all_precision /= 6
  all_recall /= 6
  all_f1 /= 6


  print(f"Precision = {all_precision}, Recall = {all_recall}, F1-score = {all_f1}")
  print("===============NEXT STEP===============")

  with open("seed_0_fcmf_model.txt",'a') as f:
    f.writelines(f"\tPrecision = {all_precision}, Recall = {all_recall}, F1-score = {all_f1} \n")

In [ ]:
def save_model(path, model, epoch):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
    },path)

In [ ]:
def load_model(path):
    check_point = torch.load(path)
    return check_point

In [ ]:
torch.manual_seed(sss)
np.random.seed(sss)

In [ ]:
best_loss = -1
for epoch in range(NUM_EPOCHS):
  train_loss = train_one_epoch(visual_model,fcmf_model,optimizer, criterion, train_loader, epoch ,device,scaler)
  eval(visual_model,fcmf_model,criterion, dev_loader, device)


In [ ]:
eval(visual_model,fcmf_model,criterion, test_loader, device)